# 03 · Statistical Tables (Supp Tables 3–9)

This notebook reproduces all seven supplementary tables from  
*Rahnev, Nature Communications 2025*.

**Requirements**: Run `02_compute_measures.ipynb` first to generate the `.npz` files in `notebooks/precomputed/`.  
Alternatively, run `analysis_core.py` directly from the command line.

| Table | Content | Dataset |
|-------|---------|--------|
| Supp Table 3 | Difficulty dependence | Shekhar (n=20) |
| Supp Table 4 | Difficulty dependence | Rouault1 (n≈466) |
| Supp Table 5 | Difficulty dependence | Rouault2 (n≈484) |
| Supp Table 6 | Metacognitive bias | Haddara (n=70) |
| Supp Table 7 | Metacognitive bias | Maniscalco (n≈22) |
| Supp Table 8 | Metacognitive bias | Shekhar (n=20) |
| Supp Table 9 | Response bias (ANOVA) | Locke (n=10) |

All t-tests are uncorrected two-sided one-sample tests against 0.

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')

REPO = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO, 'src'))
sys.path.insert(0, os.path.join(REPO, 'notebooks'))

import numpy as np
import pandas as pd
from scipy import stats
from analysis_core import (
    MEASURE_NAMES, N_MEASURES, ttest_1samp, rm_anova_1way,
    preprocess_haddara, preprocess_maniscalco, preprocess_shekhar,
    preprocess_rouault, preprocess_locke,
    compute_difficulty_shekhar, compute_difficulty_rouault,
    compute_bias, compute_bias_shekhar,
    compute_response_bias_locke,
)

OUT = os.path.join(REPO, 'notebooks', 'precomputed')

def p_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

print('Environment ready.')

Environment ready.


In [2]:
def load_or_compute(npz_path, compute_fn, *args, key=None):
    """Load from disk if available, otherwise compute and save."""
    if os.path.exists(npz_path):
        data = np.load(npz_path, allow_pickle=True)
        return data[key] if key else data
    print(f'  {os.path.basename(npz_path)} not found — computing now (may take minutes)...')
    result = compute_fn(*args)
    if key:
        np.savez(npz_path, **{key: result})
        return result
    return result

print('Loader function defined.')

Loader function defined.


## Difficulty Dependence (Supp Tables 3–5)

For each subject, all-measure values are computed separately at the highest and lowest difficulty level, then the difference (easy − hard) is tested against 0 with a one-sample t-test.

- **Shekhar**: contrast 3 (easy, ~89% acc) minus contrast 1 (hard, ~67% acc)
- **Rouault1/2**: high-DotDiff (> median) minus low-DotDiff (≤ median)

In [3]:
# ── Supp Table 3: Shekhar ──────────────────────────────────────────────────
sh = preprocess_shekhar()
sh_diff_path = os.path.join(OUT, 'shekhar_results.npz')

if os.path.exists(sh_diff_path):
    sh_diff_full = np.load(sh_diff_path)['diff']   # (n_sub, 3, N_MEASURES)
    # contrast 3 (index 2) - contrast 1 (index 0)
    sh_diff = sh_diff_full[:, [0, 2], :]           # keep hard and easy only
else:
    sh_diff = compute_difficulty_shekhar(sh)        # (n_sub, 2, N_MEASURES) [hard, easy]

delta = sh_diff[:, 1, :] - sh_diff[:, 0, :]       # easy - hard

REPORTED_T3 = {
    "meta-d'": (22.616, 5.057), 'AUC2': (20.612, 4.609), 'Gamma': (29.238, 6.538),
    'Phi': (10.898, 2.437), 'DeltaConf': (14.834, 3.317),
    'M-Ratio': (-1.240, -0.277), 'AUC2-Ratio': (-3.215, -0.719),
    'M-Diff': (-4.087, -0.914), 'AUC2-Diff': (-4.006, -0.896),
    "d'": (23.777, 5.317), 'Criterion': (0.166, 0.037), 'Confidence': (14.543, 3.252),
}

rows = []
for m, name in enumerate(MEASURE_NAMES):
    t, df, p, d, cil, ciu = ttest_1samp(delta[:, m])
    rep = REPORTED_T3.get(name, (np.nan, np.nan))
    rows.append({'Measure': name,
                 't (Python)': round(t,3) if not np.isnan(t) else np.nan,
                 't (MATLAB)': rep[0],
                 'df': int(df) if not np.isnan(df) else np.nan,
                 'p': f'{p:.3f}' if not np.isnan(p) else 'nan',
                 'sig': p_stars(p) if not np.isnan(p) else '',
                 "Cohen's d": round(d,3) if not np.isnan(d) else np.nan,
                 'd (MATLAB)': rep[1],
                 'CI lower': round(cil,3) if not np.isnan(cil) else np.nan,
                 'CI upper': round(ciu,3) if not np.isnan(ciu) else np.nan})

df_t3 = pd.DataFrame(rows)
print('Supplementary Table 3: Difficulty dependence — Shekhar (n=20)')
print('comparing contrast 3 (easy) vs contrast 1 (hard)')
print(df_t3.to_string(index=False))

Supplementary Table 3: Difficulty dependence — Shekhar (n=20)
comparing contrast 3 (easy) vs contrast 1 (hard)


         Measure  t (Python)  t (MATLAB)   df     p sig  Cohen's d  d (MATLAB)  CI lower  CI upper
         meta-d'      22.732      22.616 19.0 0.000 ***      5.083       5.057     1.214     1.460
            AUC2      20.728      20.612 19.0 0.000 ***      4.635       4.609     0.150     0.183
           Gamma      29.393      29.238 19.0 0.000 ***      6.572       6.538     0.352     0.406
             Phi      10.926      10.898 19.0 0.000 ***      2.443       2.437     0.103     0.152
       DeltaConf      14.909      14.834 19.0 0.000 ***      3.334       3.317     0.911     1.208
         M-Ratio      -1.168      -1.240 19.0 0.257  ns     -0.261      -0.277    -0.162     0.046
      AUC2-Ratio      -3.015      -3.215 19.0 0.007  **     -0.674      -0.719    -0.056    -0.010
     Gamma-Ratio      -0.408         NaN 19.0 0.688  ns     -0.091         NaN    -0.125     0.084
       Phi-Ratio      -0.862         NaN 19.0 0.400  ns     -0.193         NaN    -0.148     0.062
 DeltaConf

In [4]:
# ── Supp Table 4: Rouault1 ────────────────────────────────────────────────
r1_diff_path = os.path.join(OUT, 'rouault1_results.npz')

if os.path.exists(r1_diff_path):
    r1_diff = np.load(r1_diff_path)['diff']   # (n_sub, 2, N_MEASURES)
else:
    r1 = preprocess_rouault(1)
    r1_diff = compute_difficulty_rouault(r1)

delta = r1_diff[:, 1, :] - r1_diff[:, 0, :]   # high - low contrast

REPORTED_T4 = {
    "meta-d'": (35.285, 1.654), 'AUC2': (35.405, 1.644), "d'": (49.278, 2.288),
    'Confidence': (32.390, 1.505),
}

rows = []
for m, name in enumerate(MEASURE_NAMES):
    t, df, p, d, cil, ciu = ttest_1samp(delta[:, m])
    rep = REPORTED_T4.get(name, (np.nan, np.nan))
    rows.append({'Measure': name,
                 't (Python)': round(t,3) if not np.isnan(t) else np.nan,
                 't (MATLAB)': rep[0],
                 'df': int(df) if not np.isnan(df) else np.nan,
                 'p': f'{p:.4f}' if not np.isnan(p) else 'nan',
                 'sig': p_stars(p) if not np.isnan(p) else '',
                 "Cohen's d": round(d,3) if not np.isnan(d) else np.nan,
                 'CI lower': round(cil,3) if not np.isnan(cil) else np.nan,
                 'CI upper': round(ciu,3) if not np.isnan(ciu) else np.nan})

df_t4 = pd.DataFrame(rows)
print('Supplementary Table 4: Difficulty dependence — Rouault1')
print(df_t4.to_string(index=False))

Supplementary Table 4: Difficulty dependence — Rouault1
         Measure  t (Python)  t (MATLAB)    df      p sig  Cohen's d  CI lower  CI upper
         meta-d'      35.388      35.285 465.0 0.0000 ***      1.639     1.073     1.200
            AUC2      35.459      35.405 465.0 0.0000 ***      1.643     0.134     0.149
           Gamma      34.244         NaN 465.0 0.0000 ***      1.586     0.330     0.370
             Phi      24.368         NaN 465.0 0.0000 ***      1.129     0.131     0.154
       DeltaConf      31.123         NaN 465.0 0.0000 ***      1.442     0.688     0.781
         M-Ratio      -1.452         NaN 445.0 0.1472  ns     -0.069    -0.134     0.020
      AUC2-Ratio      -4.443         NaN 465.0 0.0000 ***     -0.206    -0.043    -0.017
     Gamma-Ratio      -1.234         NaN 465.0 0.2179  ns     -0.057    -1.893     0.433
       Phi-Ratio       0.208         NaN 465.0 0.8355  ns      0.010    -1.029     1.272
 DeltaConf-Ratio       0.139         NaN 465.0 0.8894 

In [5]:
# ── Supp Table 5: Rouault2 ────────────────────────────────────────────────
r2_diff_path = os.path.join(OUT, 'rouault2_results.npz')

if os.path.exists(r2_diff_path):
    r2_diff = np.load(r2_diff_path)['diff']
else:
    r2 = preprocess_rouault(2)
    r2_diff = compute_difficulty_rouault(r2)

delta = r2_diff[:, 1, :] - r2_diff[:, 0, :]

REPORTED_T5 = {
    "meta-d'": (15.304, 0.701), 'AUC2': (13.657, 0.623), "d'": (48.583, 2.241),
    'Confidence': (15.334, 0.702),
}

rows = []
for m, name in enumerate(MEASURE_NAMES):
    t, df, p, d, cil, ciu = ttest_1samp(delta[:, m])
    rep = REPORTED_T5.get(name, (np.nan, np.nan))
    rows.append({'Measure': name,
                 't (Python)': round(t,3) if not np.isnan(t) else np.nan,
                 't (MATLAB)': rep[0],
                 'df': int(df) if not np.isnan(df) else np.nan,
                 'p': f'{p:.4f}' if not np.isnan(p) else 'nan',
                 'sig': p_stars(p) if not np.isnan(p) else '',
                 "Cohen's d": round(d,3) if not np.isnan(d) else np.nan,
                 'CI lower': round(cil,3) if not np.isnan(cil) else np.nan,
                 'CI upper': round(ciu,3) if not np.isnan(ciu) else np.nan})

df_t5 = pd.DataFrame(rows)
print('Supplementary Table 5: Difficulty dependence — Rouault2')
print(df_t5.to_string(index=False))

Supplementary Table 5: Difficulty dependence — Rouault2
         Measure  t (Python)  t (MATLAB)    df      p sig  Cohen's d  CI lower  CI upper
         meta-d'      16.076      15.304 483.0 0.0000 ***      0.731     0.435     0.556
            AUC2      13.705      13.657 483.0 0.0000 ***      0.623     0.049     0.066
           Gamma      12.162         NaN 483.0 0.0000 ***      0.553     0.111     0.154
             Phi       9.326         NaN 483.0 0.0000 ***      0.424     0.049     0.075
       DeltaConf      13.366         NaN 483.0 0.0000 ***      0.608     0.235     0.316
         M-Ratio      -3.850         NaN 483.0 0.0001 ***     -0.175    -0.148    -0.048
      AUC2-Ratio      -5.888         NaN 483.0 0.0000 ***     -0.268    -0.054    -0.027
     Gamma-Ratio      -3.461         NaN 483.0 0.0006 ***     -0.157    -0.140    -0.039
       Phi-Ratio      -3.380         NaN 483.0 0.0008 ***     -0.154    -0.154    -0.041
 DeltaConf-Ratio      -4.099         NaN 483.0 0.0000 

## Metacognitive Bias (Supp Tables 6–8)

For each subject, all measures are computed after two Xue et al. (2021) recodings:  
- **Recode 1** (high-conf bias): removes lowest rating  
- **Recode 2** (low-conf bias): removes highest rating  

Test: recode2 − recode1.  A significant effect means the measure is confounded by metacognitive bias.

> Note: d' and Criterion are excluded (Xue recoding only affects confidence, not responses).

In [6]:
# ── Supp Table 6: Haddara ─────────────────────────────────────────────────
ha_path = os.path.join(OUT, 'haddara_results.npz')

if os.path.exists(ha_path):
    ha_bias = np.load(ha_path)['bias']   # (n_sub, 2, N_MEASURES)
else:
    ha = preprocess_haddara()
    ha_bias = compute_bias(ha)

delta = ha_bias[:, 1, :] - ha_bias[:, 0, :]   # recode2 - recode1

REPORTED_T6 = {
    "meta-d'": (2.584, 0.309), 'AUC2': (0.688, 0.082), 'Gamma': (-4.331, -0.518),
    'Phi': (1.257, 0.150), 'DeltaConf': (1.034, 0.124),
    'M-Ratio': (1.994, 0.238), 'Gamma-Diff': (2.334, 0.279),
    'meta-noise': (-1.072, -0.128), 'meta-uncertainty': (1.956, 0.234),
    'Confidence': (24.538, 2.933),
}

EXCLUDE = {"d'", "Criterion"}
rows = []
for m, name in enumerate(MEASURE_NAMES):
    if name in EXCLUDE:
        continue
    t, df, p, d, cil, ciu = ttest_1samp(delta[:, m])
    rep = REPORTED_T6.get(name, (np.nan, np.nan))
    rows.append({'Measure': name,
                 't (Python)': round(t,3) if not np.isnan(t) else np.nan,
                 't (MATLAB)': rep[0],
                 'df': int(df) if not np.isnan(df) else np.nan,
                 'p': f'{p:.3f}' if not np.isnan(p) else 'nan',
                 'sig': p_stars(p) if not np.isnan(p) else '',
                 "Cohen's d": round(d,3) if not np.isnan(d) else np.nan,
                 'CI lower': round(cil,3) if not np.isnan(cil) else np.nan,
                 'CI upper': round(ciu,3) if not np.isnan(ciu) else np.nan})

df_t6 = pd.DataFrame(rows)
print('Supplementary Table 6: Metacognitive bias — Haddara (n=70)')
print(df_t6.to_string(index=False))

Supplementary Table 6: Metacognitive bias — Haddara (n=70)
         Measure  t (Python)  t (MATLAB)   df     p sig  Cohen's d  CI lower  CI upper
         meta-d'       2.318       2.584 69.0 0.023   *      0.277     0.008     0.102
            AUC2       0.688       0.688 69.0 0.494  ns      0.082    -0.008     0.016
           Gamma      -4.331      -4.331 69.0 0.000 ***     -0.518    -0.068    -0.025
             Phi       1.257       1.257 69.0 0.213  ns      0.150    -0.005     0.021
       DeltaConf       1.034       1.034 69.0 0.305  ns      0.124    -0.019     0.059
         M-Ratio       1.795       1.994 69.0 0.077  ns      0.215    -0.004     0.067
      AUC2-Ratio       1.176         NaN 69.0 0.244  ns      0.141    -0.003     0.013
     Gamma-Ratio       0.510         NaN 69.0 0.612  ns      0.061    -0.029     0.049
       Phi-Ratio       1.062         NaN 69.0 0.292  ns      0.127    -0.020     0.065
 DeltaConf-Ratio       1.664         NaN 69.0 0.101  ns      0.199    -

In [7]:
# ── Supp Table 7: Maniscalco ──────────────────────────────────────────────
ma_path = os.path.join(OUT, 'maniscalco_results.npz')

if os.path.exists(ma_path):
    ma_bias = np.load(ma_path)['bias']
else:
    ma = preprocess_maniscalco()
    ma_bias = compute_bias(ma)

delta = ma_bias[:, 1, :] - ma_bias[:, 0, :]

REPORTED_T7 = {
    "meta-d'": (2.711, 0.578), 'AUC2': (3.794, 0.809), 'Phi': (5.262, 1.122),
    'DeltaConf': (5.242, 1.118), 'Confidence': (17.328, 3.694),
}

rows = []
for m, name in enumerate(MEASURE_NAMES):
    if name in EXCLUDE:
        continue
    t, df, p, d, cil, ciu = ttest_1samp(delta[:, m])
    rep = REPORTED_T7.get(name, (np.nan, np.nan))
    rows.append({'Measure': name,
                 't (Python)': round(t,3) if not np.isnan(t) else np.nan,
                 't (MATLAB)': rep[0],
                 'df': int(df) if not np.isnan(df) else np.nan,
                 'p': f'{p:.3f}' if not np.isnan(p) else 'nan',
                 'sig': p_stars(p) if not np.isnan(p) else '',
                 "Cohen's d": round(d,3) if not np.isnan(d) else np.nan,
                 'CI lower': round(cil,3) if not np.isnan(cil) else np.nan,
                 'CI upper': round(ciu,3) if not np.isnan(ciu) else np.nan})

df_t7 = pd.DataFrame(rows)
print('Supplementary Table 7: Metacognitive bias — Maniscalco (n≈22)')
print(df_t7.to_string(index=False))

Supplementary Table 7: Metacognitive bias — Maniscalco (n≈22)


         Measure  t (Python)  t (MATLAB)   df     p sig  Cohen's d  CI lower  CI upper
         meta-d'       2.255       2.711 21.0 0.035   *      0.481     0.012     0.303
            AUC2       3.794       3.794 21.0 0.001  **      0.809     0.016     0.054
           Gamma      -1.646         NaN 21.0 0.115  ns     -0.351    -0.126     0.015
             Phi       5.262       5.262 21.0 0.000 ***      1.122     0.040     0.091
       DeltaConf       5.242       5.242 21.0 0.000 ***      1.118     0.107     0.247
         M-Ratio       0.601         NaN 21.0 0.554  ns      0.128    -0.065     0.118
      AUC2-Ratio       1.500         NaN 21.0 0.149  ns      0.320    -0.005     0.032
     Gamma-Ratio      -0.130         NaN 21.0 0.898  ns     -0.028    -0.081     0.072
       Phi-Ratio       0.692         NaN 21.0 0.496  ns      0.148    -0.055     0.110
 DeltaConf-Ratio       1.841         NaN 21.0 0.080  ns      0.393    -0.009     0.143
          M-Diff       2.249         NaN 21

In [8]:
# ── Supp Table 8: Shekhar ─────────────────────────────────────────────────
sh_path = os.path.join(OUT, 'shekhar_results.npz')

if os.path.exists(sh_path):
    sh_bias_full = np.load(sh_path)['bias']   # (n_sub, 3, 2, N_MEASURES)
    sh_bias = np.nanmean(sh_bias_full, axis=1)  # average over contrasts → (n_sub, 2, N_MEASURES)
else:
    sh = preprocess_shekhar()
    sh_bias_arr = compute_bias_shekhar(sh)       # (n_sub, 2, N_MEASURES)
    sh_bias = sh_bias_arr

delta = sh_bias[:, 1, :] - sh_bias[:, 0, :]

REPORTED_T8 = {
    'AUC2': (2.804, 0.627), 'Gamma': (-4.284, -0.958), 'Phi': (5.133, 1.148),
    'DeltaConf-Ratio': (2.992, 0.669), 'Confidence': (13.845, 3.096),
}

rows = []
for m, name in enumerate(MEASURE_NAMES):
    if name in EXCLUDE:
        continue
    t, df, p, d, cil, ciu = ttest_1samp(delta[:, m])
    rep = REPORTED_T8.get(name, (np.nan, np.nan))
    rows.append({'Measure': name,
                 't (Python)': round(t,3) if not np.isnan(t) else np.nan,
                 't (MATLAB)': rep[0],
                 'df': int(df) if not np.isnan(df) else np.nan,
                 'p': f'{p:.3f}' if not np.isnan(p) else 'nan',
                 'sig': p_stars(p) if not np.isnan(p) else '',
                 "Cohen's d": round(d,3) if not np.isnan(d) else np.nan,
                 'CI lower': round(cil,3) if not np.isnan(cil) else np.nan,
                 'CI upper': round(ciu,3) if not np.isnan(ciu) else np.nan})

df_t8 = pd.DataFrame(rows)
print('Supplementary Table 8: Metacognitive bias — Shekhar (n=20)')
print(df_t8.to_string(index=False))

Supplementary Table 8: Metacognitive bias — Shekhar (n=20)
         Measure  t (Python)  t (MATLAB)   df     p sig  Cohen's d  CI lower  CI upper
         meta-d'       0.425         NaN 19.0 0.675  ns      0.095    -0.019     0.029
            AUC2       2.804       2.804 19.0 0.011   *      0.627     0.004     0.027
           Gamma      -4.281      -4.284 19.0 0.000 ***     -0.957    -0.055    -0.019
             Phi       5.138       5.133 19.0 0.000 ***      1.149     0.015     0.036
       DeltaConf       1.747         NaN 19.0 0.097  ns      0.391    -0.013     0.140
         M-Ratio       0.605         NaN 19.0 0.553  ns      0.135    -0.010     0.018
      AUC2-Ratio      -0.872         NaN 19.0 0.394  ns     -0.195    -0.009     0.004
     Gamma-Ratio      -0.227         NaN 19.0 0.823  ns     -0.051    -0.017     0.014
       Phi-Ratio       1.847         NaN 19.0 0.080  ns      0.413    -0.002     0.025
 DeltaConf-Ratio       2.954       2.992 19.0 0.008  **      0.661     

## Response Bias (Supp Table 9) — Locke 2020

One-way repeated-measures ANOVA across 7 conditions.  
`F(6, 54)` → 10 subjects × 7 conditions.

The paper reports no significant effect for any metacognitive measure — only Criterion shows a significant condition effect, confirming that response bias does not contaminate well-designed metacognitive measures.

In [9]:
lo_path = os.path.join(OUT, 'locke_results.npz')

if os.path.exists(lo_path):
    lo_rb = np.load(lo_path)['rb']   # (n_sub, 7, N_MEASURES)
else:
    lo = preprocess_locke()
    lo_rb = compute_response_bias_locke(lo)

REPORTED_T9 = {
    "meta-d'": (1.472, 0.205, 0.141), 'AUC2': (0.742, 0.618, 0.076),
    'Criterion': (12.185, 0.001, 0.575), 'Confidence': (0.482, 0.819, 0.051),
}

rows = []
for m, name in enumerate(MEASURE_NAMES):
    data = lo_rb[:, :, m]  # (n_sub, 7)
    complete = ~np.any(np.isnan(data), axis=1)
    data_c = data[complete]
    if data_c.shape[0] < 2:
        rows.append({'Measure': name, 'F(6,54)': np.nan, 'p': np.nan, 'η²p': np.nan, 'sig': ''})
        continue
    F, df_b, df_e, p, eta2p = rm_anova_1way(data_c)
    rep = REPORTED_T9.get(name, (np.nan, np.nan, np.nan))
    rows.append({'Measure': name,
                 f'F({df_b},{df_e}) Python': round(F,3),
                 'F(6,54) MATLAB': rep[0],
                 'p': f'{p:.3f}',
                 'sig': p_stars(p),
                 'η²p': round(eta2p,3),
                 'η²p MATLAB': rep[2]})

df_t9 = pd.DataFrame(rows)
print('Supplementary Table 9: Response bias — Locke (n=10, 7 conditions)')
print(df_t9.to_string(index=False))

Supplementary Table 9: Response bias — Locke (n=10, 7 conditions)
         Measure  F(6,54) Python  F(6,54) MATLAB     p sig   η²p  η²p MATLAB  F(6,54)
         meta-d'           1.472           1.472 0.205  ns 0.141       0.141      NaN
            AUC2           0.742           0.742 0.618  ns 0.076       0.076      NaN
           Gamma           0.863             NaN 0.528  ns 0.087         NaN      NaN
             Phi           0.927             NaN 0.483  ns 0.093         NaN      NaN
       DeltaConf           0.742             NaN 0.618  ns 0.076         NaN      NaN
         M-Ratio           1.090             NaN 0.380  ns 0.108         NaN      NaN
      AUC2-Ratio           1.176             NaN 0.333  ns 0.116         NaN      NaN
     Gamma-Ratio           1.063             NaN 0.396  ns 0.106         NaN      NaN
       Phi-Ratio           1.078             NaN 0.387  ns 0.107         NaN      NaN
 DeltaConf-Ratio           1.072             NaN 0.391  ns 0.106         N

## Validation Summary

The cell below computes match scores between Python and MATLAB t-values.

In [10]:
def match_score(df, py_col, matlab_col, tol=0.05):
    """Fraction of rows where |Python - MATLAB| / |MATLAB| < tol."""
    rows = df.dropna(subset=[py_col, matlab_col])
    rows = rows[rows[matlab_col] != 0]
    err = abs(rows[py_col] - rows[matlab_col]) / abs(rows[matlab_col])
    good = (err < tol).mean()
    return good, err.mean()

print('Table | Match (5% tol) | Mean rel. error')
print('-' * 45)
for label, df, pcol, mcol in [
    ('T3 Shekhar diff',   df_t3, 't (Python)', 't (MATLAB)'),
    ('T4 Rouault1 diff',  df_t4, 't (Python)', 't (MATLAB)'),
    ('T5 Rouault2 diff',  df_t5, 't (Python)', 't (MATLAB)'),
    ('T6 Haddara bias',   df_t6, 't (Python)', 't (MATLAB)'),
    ('T7 Maniscalco bias',df_t7, 't (Python)', 't (MATLAB)'),
    ('T8 Shekhar bias',   df_t8, 't (Python)', 't (MATLAB)'),
]:
    frac, err = match_score(df, pcol, mcol)
    print(f'{label:<22}  {frac*100:5.1f}%          {err*100:5.1f}%')

Table | Match (5% tol) | Mean rel. error
---------------------------------------------
T3 Shekhar diff          83.3%            1.8%
T4 Rouault1 diff        100.0%            0.3%
T5 Rouault2 diff         75.0%            2.7%
T6 Haddara bias          75.0%            2.7%
T7 Maniscalco bias       80.0%            3.4%
T8 Shekhar bias         100.0%            0.3%
